In [1]:
!pip install -q rasterio

In [2]:
# ============================================================
# DATASET PREPARATION FOR INDIAN ROAD FINE-TUNING
# Does three things:
#   1. Flattens the per-area directory structure into flat train/val dirs
#   2. Splits by AREA (not by tile) to prevent data leakage from overlapping tiles
#   3. Optionally re-chips original TIFFs at smaller stride for more pairs
# ============================================================

# !pip install -q rasterio

import os
import shutil
import random
import numpy as np
import cv2
import rasterio
from pathlib import Path
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================

# Root containing all 12 area folders (each with images/ and masks/)
DATASET_ROOT = "/kaggle/input/datasets/amrenderpalarwal/bangalore-roads/Bengaluru_Chipped_Districts/"

# Where to write the final flat train/val structure
OUTPUT_ROOT = "/kaggle/working/"

# Areas to hold out for validation — choose 2 diverse ones:
# Whitefield (IT corridor, modern roads) + Hebbal (mixed) gives good diversity
VAL_AREAS = ["Whitefield", "Hebbal"]

# Tile settings — change these if re-chipping from TIFFs
TILE_SIZE = 1024
CHIP_STRIDE = 512       # current stride (= 50% overlap between tiles)

# To generate more chips from existing TIFFs, set RECHIP_FROM_TIFFS=True
RECHIP_FROM_TIFFS = True
TIFF_ROOT = "/kaggle/input/datasets/amrenderpalarwal/isro-tiffs/ISRO/"    
NEW_STRIDE = 256        # smaller stride = more chips (4× more than stride=512)

MAX_NODATA_PCT = 0.20   # skip tiles with >20% black padding

# ============================================================
# 1. AUDIT THE EXISTING DATASET
# ============================================================

def audit_dataset(root):
    root = Path(root)
    print("=" * 55)
    print("DATASET AUDIT")
    print("=" * 55)

    total_images = 0
    total_masks  = 0
    area_counts  = {}

    for area_dir in sorted(root.iterdir()):
        if not area_dir.is_dir():
            continue
        img_dir  = area_dir / "images"
        mask_dir = area_dir / "masks"
        if not img_dir.exists():
            continue

        imgs  = sorted(img_dir.glob("*.jpg")) + sorted(img_dir.glob("*.png"))
        masks = sorted(mask_dir.glob("*.png")) if mask_dir.exists() else []

        area_counts[area_dir.name] = len(imgs)
        total_images += len(imgs)
        total_masks  += len(masks)
        paired = "✓" if len(imgs) == len(masks) else f"⚠ MISMATCH ({len(masks)} masks)"
        print(f"  {area_dir.name:<20} {len(imgs):>4} images   {paired}")

    print("-" * 55)
    print(f"  TOTAL:               {total_images:>4} images | {total_masks} masks")
    print(f"  Areas found:         {len(area_counts)}")
    print()

    if RECHIP_FROM_TIFFS:
        ratio = (TILE_SIZE / NEW_STRIDE) ** 2
        print(f"\n  Re-chipping at stride={NEW_STRIDE}px would give "
              f"~{int(total_images * ratio)} tiles ({ratio:.1f}× more)")

    return area_counts

# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def nodata_pct(img_path):
    img = cv2.imread(str(img_path))
    if img is None:
        return 1.0
    black = ((img[:,:,0]==0) & (img[:,:,1]==0) & (img[:,:,2]==0))
    return float(black.mean())

def read_tiff(path, is_mask=False):
    """
    Safely reads 32-bit/16-bit GeoTIFFs using Rasterio and normalizes
    them to 8-bit so OpenCV can save them correctly.
    """
    with rasterio.open(str(path)) as src:
        img = src.read()
        img = np.transpose(img, (1, 2, 0)) # Convert (C, H, W) to (H, W, C)
        
        if is_mask:
            if img.ndim == 3 and img.shape[2] >= 1:
                img = img[:, :, 0] # Extract just the first channel for masks
            else:
                img = np.squeeze(img)
        
        # Normalize 32-bit floats or 16-bit ints to 8-bit uint8
        if img.dtype != np.uint8:
            if img.dtype in [np.float32, np.float64] and img.max() <= 1.0:
                img = (img * 255.0).astype(np.uint8)
            else:
                # Min-max scale to 0-255 safely
                img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
        
        # Rasterio reads as RGB, but OpenCV imwrite expects BGR. Swap channels!
        if not is_mask and img.ndim == 3 and img.shape[2] == 3:
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            
        return img

# ============================================================
# 3. FLATTEN + AREA-BASED SPLIT (For existing chips)
# ============================================================

def prepare_flat_dataset(root, output_root, val_areas, max_nodata=0.20):
    root        = Path(root)
    output_root = Path(output_root)

    for split in ("train", "val"):
        (output_root / split / "images").mkdir(parents=True, exist_ok=True)
        (output_root / split / "masks").mkdir(parents=True, exist_ok=True)

    train_count = val_count = skipped = 0

    for area_dir in sorted(root.iterdir()):
        if not area_dir.is_dir():
            continue

        img_dir  = area_dir / "images"
        mask_dir = area_dir / "masks"
        if not img_dir.exists() or not mask_dir.exists():
            continue

        split = "val" if area_dir.name in val_areas else "train"
        imgs  = sorted(img_dir.glob("*.jpg")) + sorted(img_dir.glob("*.png"))

        for img_path in tqdm(imgs, desc=f"{area_dir.name} → {split}"):
            mask_path = mask_dir / img_path.name.replace("_sat.jpg", "_mask.png").replace(".jpg", ".png")
            if not mask_path.exists():
                mask_path = mask_dir / (img_path.stem + "_mask.png")
            if not mask_path.exists():
                skipped += 1
                continue

            if nodata_pct(img_path) > max_nodata:
                skipped += 1
                continue

            prefix   = f"{area_dir.name}_"
            dst_img  = output_root / split / "images" / (prefix + img_path.name)
            dst_mask = output_root / split / "masks"  / (prefix + mask_path.name)

            shutil.copy(img_path,  dst_img)
            shutil.copy(mask_path, dst_mask)

            if split == "train": train_count += 1
            else: val_count += 1

    print(f"\nDataset prepared: Train: {train_count} | Val: {val_count} | Skipped: {skipped}")
    return train_count, val_count

# ============================================================
# 4. RE-CHIP FROM ORIGINAL TIFFS
# ============================================================

def rechip_tiff_pair(img_tiff, mask_tiff, out_img_dir, out_mask_dir,
                     area_name, tile_size=1024, stride=256,
                     max_nodata=0.20):
    try:
        # Replaced cv2.imread with our robust Rasterio function
        img = read_tiff(img_tiff, is_mask=False)
        msk = read_tiff(mask_tiff, is_mask=True)
    except Exception as e:
        print(f"  ⚠ Could not load {img_tiff.name}: {e}")
        return 0

    H, W = img.shape[:2]
    count = 0

    for y in range(0, H - tile_size + 1, stride):
        for x in range(0, W - tile_size + 1, stride):
            chip_img  = img [y:y+tile_size, x:x+tile_size]
            chip_mask = msk [y:y+tile_size, x:x+tile_size]

            # Check for black padding safely depending on channel config
            if chip_img.ndim == 3:
                black_pct = ((chip_img[:,:,0]==0) & (chip_img[:,:,1]==0) & (chip_img[:,:,2]==0)).mean()
            else:
                black_pct = (chip_img == 0).mean()
                
            if black_pct > max_nodata:
                continue

            fname = f"{area_name}_{y}_{x}_sat.jpg"
            mname = f"{area_name}_{y}_{x}_mask.png"
            cv2.imwrite(str(out_img_dir / fname), chip_img,
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            cv2.imwrite(str(out_mask_dir / mname), chip_mask)
            count += 1

    return count

def rechip_all_tiffs(tiff_root, output_root, val_areas,
                     tile_size=1024, stride=256, max_nodata=0.20):
    tiff_root   = Path(tiff_root)
    output_root = Path(output_root)

    for split in ("train", "val"):
        (output_root / split / "images").mkdir(parents=True, exist_ok=True)
        (output_root / split / "masks").mkdir(parents=True, exist_ok=True)

    total = 0
    for area_dir in sorted(tiff_root.iterdir()):
        if not area_dir.is_dir():
            continue
        split = "val" if area_dir.name in val_areas else "train"

        img_tiff  = None
        mask_tiff = None
        for f in area_dir.glob("*.tif"):
            if "mask" in f.name.lower() or "label" in f.name.lower():
                mask_tiff = f
            else:
                img_tiff  = f

        if img_tiff is None or mask_tiff is None:
            print(f"  ⚠ {area_dir.name}: couldn't find image+mask TIF pair")
            continue

        n = rechip_tiff_pair(
            img_tiff, mask_tiff,
            output_root / split / "images",
            output_root / split / "masks",
            area_name=area_dir.name,
            tile_size=tile_size,
            stride=stride,
            max_nodata=max_nodata,
        )
        total += n
        print(f"  {area_dir.name} ({split}): {n} chips generated")

    print(f"\nTotal chips generated: {total}")
    return total

# ============================================================
# 5. ENTRY POINT
# ============================================================

if __name__ == "__main__":
    audit_dataset(DATASET_ROOT)

    if RECHIP_FROM_TIFFS:
        print("\n=== RE-CHIPPING FROM ORIGINAL TIFFS ===")
        rechip_all_tiffs(
            tiff_root=TIFF_ROOT,
            output_root=OUTPUT_ROOT,
            val_areas=VAL_AREAS,
            tile_size=TILE_SIZE,
            stride=NEW_STRIDE,
            max_nodata=MAX_NODATA_PCT,
        )
    else:
        print("\n=== FLATTENING EXISTING CHIPS ===")
        prepare_flat_dataset(
            root=DATASET_ROOT,
            output_root=OUTPUT_ROOT,
            val_areas=VAL_AREAS,
            max_nodata=MAX_NODATA_PCT,
        )

    print(f"\nOutput ready at: {OUTPUT_ROOT}")

DATASET AUDIT
  Basavanagudi            6 images   ✓
  Bellandur              16 images   ✓
  Hebbal                  4 images   ✓
  Jayanagar               3 images   ✓
  Madiwala                4 images   ✓
  Malleshwaram           12 images   ✓
  Shivajinagar            6 images   ✓
  Whitefield             34 images   ✓
  Yelahanka              15 images   ✓
  chickpet                2 images   ✓
  indira                 25 images   ✓
  koramangala            34 images   ✓
-------------------------------------------------------
  TOTAL:                161 images | 161 masks
  Areas found:         12


  Re-chipping at stride=256px would give ~2576 tiles (16.0× more)

=== RE-CHIPPING FROM ORIGINAL TIFFS ===
  Basavanagudi (train): 4 chips generated
  Bellandur (train): 30 chips generated
  Hebbal (val): 4 chips generated
  Jayanagar (train): 0 chips generated
  Madiwala (train): 2 chips generated
  Malleshwaram (train): 18 chips generated
  Shivajinagar (train): 6 chips generated
  

In [3]:
!pip install -q segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 

In [4]:
# ============================================================
# SKELETONIZE GROUND TRUTH MASKS
# Thins thick QGIS polygon road fills (~15px) to 1-3px centerlines
# so they match the model's prediction style and dice scores are meaningful.
#
# Run this ONCE on your mask directories before fine-tuning.
# Creates new *_skel directories — originals are untouched.
# ============================================================

import os
import cv2
import numpy as np
from pathlib import Path
from skimage.morphology import skeletonize, thin
from tqdm import tqdm

# ============================================================
# CONFIG — point at your flat train/val mask directories
# ============================================================
MASK_DIRS = [
    "/kaggle/working/train/masks/",
    "/kaggle/working/val/masks/",
]

# How many pixels to dilate the skeleton back out after thinning.
# 0 = pure 1px skeleton (mathematically correct but hard to learn)
# 2 = 5px wide result (easier for the model to predict, still much thinner than 15px)
# 3 = 7px wide result (recommended starting point)
DILATE_PX = 3

# ============================================================
# CORE FUNCTION
# ============================================================
def skeletonize_mask(mask_path, dilate_px=3):
    """
    Loads a binary mask PNG, skeletonizes it, optionally dilates back,
    returns result as uint8 (0 or 255).

    Why skeletonize then re-dilate slightly?
      - Pure 1px skeleton is too thin for the model to reliably predict
        (a 1px miss scores 0 dice even if you detected the right road)
      - Dilating back to ~5-7px gives the model a small spatial tolerance
        while still being far thinner than the original 15px fill
    """
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None

    # Binarize robustly regardless of 0/1 or 0/255 style
    binary = (mask > 127).astype(np.uint8)

    # Skeletonize — reduces to 1px wide centerline
    skel = skeletonize(binary.astype(bool)).astype(np.uint8) * 255

    # Dilate back slightly for learnability
    if dilate_px > 0:
        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE, (2*dilate_px+1, 2*dilate_px+1))
        skel = cv2.dilate(skel, kernel, iterations=1)

    return skel


def process_directory(mask_dir, dilate_px=3):
    mask_dir = Path(mask_dir)

    # Output goes into a sibling _skel directory
    out_dir = mask_dir.parent / (mask_dir.name + "_skel")
    out_dir.mkdir(exist_ok=True)

    masks = sorted(mask_dir.glob("*.png"))
    if not masks:
        print(f"  ⚠  No PNG masks found in {mask_dir}")
        return

    total_road_before = total_road_after = 0

    for mask_path in tqdm(masks, desc=str(mask_dir.name)):
        skel = skeletonize_mask(mask_path, dilate_px=dilate_px)
        if skel is None:
            continue

        # Stats for the first few files
        orig = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        total_road_before += (orig > 127).sum()
        total_road_after  += (skel > 127).sum()

        cv2.imwrite(str(out_dir / mask_path.name), skel)

    pct = 100 * total_road_after / (total_road_before + 1e-8)
    print(f"  {mask_dir.name}: {len(masks)} masks processed")
    print(f"  Road pixels: {total_road_before:,} → {total_road_after:,} "
          f"({pct:.1f}% of original width)")
    print(f"  Output: {out_dir}")
    return out_dir


# ============================================================
# QUICK VISUAL CHECK
# ============================================================
def save_comparison(mask_dir, out_dir, n=4):
    """Save side-by-side: original | skeleton for the first n masks."""
    mask_dir = Path(mask_dir)
    out_dir  = Path(out_dir)
    masks    = sorted(mask_dir.glob("*.png"))[:n]

    panels = []
    for mp in masks:
        orig = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
        skel = cv2.imread(str(out_dir / mp.name), cv2.IMREAD_GRAYSCALE)
        if orig is None or skel is None:
            continue
        orig_small = cv2.resize(orig, (512, 512))
        skel_small = cv2.resize(skel, (512, 512))
        panel = np.concatenate([orig_small, skel_small], axis=1)
        panels.append(panel)

    if panels:
        combined = np.concatenate(panels, axis=0)
        out_path = "/kaggle/working/skeleton_comparison.png"
        cv2.imwrite(out_path, combined)
        print(f"\n✅  Visual comparison saved to {out_path}")
        print("    Left = original thick mask | Right = skeletonized mask")


# ============================================================
# ENTRY POINT
# ============================================================
if __name__ == "__main__":
    print(f"Skeletonizing masks with dilate_px={DILATE_PX} "
          f"(target road width: ~{2*DILATE_PX+1}px)\n")

    skel_dirs = []
    for mask_dir in MASK_DIRS:
        print(f"\nProcessing: {mask_dir}")
        out_dir = process_directory(mask_dir, dilate_px=DILATE_PX)
        if out_dir:
            skel_dirs.append(out_dir)

    # Save a visual comparison so you can confirm it looks right
    if skel_dirs:
        save_comparison(MASK_DIRS[1], skel_dirs[1], n=4)

    print("\n" + "="*55)
    print("Now update finetune_indian.py mask paths to:")
    for d in skel_dirs:
        print(f"  '{d}/'")
    print("\nExpected dice improvement: 0.29 → 0.55+ on same data")

Skeletonizing masks with dilate_px=3 (target road width: ~7px)


Processing: /kaggle/working/train/masks/


masks: 100%|██████████| 229/229 [00:14<00:00, 15.44it/s]


  masks: 229 masks processed
  Road pixels: 14,686,943 → 17,262,557 (117.5% of original width)
  Output: /kaggle/working/train/masks_skel

Processing: /kaggle/working/val/masks/


masks: 100%|██████████| 81/81 [00:03<00:00, 25.66it/s]

  masks: 81 masks processed
  Road pixels: 4,656,119 → 5,503,574 (118.2% of original width)
  Output: /kaggle/working/val/masks_skel

✅  Visual comparison saved to /kaggle/working/skeleton_comparison.png
    Left = original thick mask | Right = skeletonized mask

Now update finetune_indian.py mask paths to:
  '/kaggle/working/train/masks_skel/'
  '/kaggle/working/val/masks_skel/'

Expected dice improvement: 0.29 → 0.55+ on same data


In [5]:
# ============================================================
# FINE-TUNING ON INDIAN DATASET  (v2 — fixed LRs + mask thinning)
# Handles: black nodata padding, thick QGIS road masks,
#          loading from existing DeepGlobe checkpoint
# ============================================================
# !pip install -q segmentation-models-pytorch

import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import segmentation_models_pytorch as smp

# ============================================================
# CONFIG
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ENCODER_NAME    = "resnet34"
ENCODER_WEIGHTS = "imagenet"

TRAIN_IMG_DIR  = "/kaggle/working/train/images/"
TRAIN_MASK_DIR = "/kaggle/working/train/masks/"
VAL_IMG_DIR    = "/kaggle/working/val/images/"
VAL_MASK_DIR   = "/kaggle/working/val/masks/"

PRETRAINED_CKPT = "/kaggle/input/notebooks/amrenderpalarwal/road-resilience-model-pretrained-encoder-unet-resn/my_checkpoint.pth.tar"

# FIX v2: increased LRs — previous 1e-6/5e-5 were too small for the
# DeepGlobe→Indian domain gap. 210 gradient steps at 1e-6 barely moved
# the weights (dice 0.2971 → 0.2995 after 10 epochs = completely flat).
ENCODER_LR = 5e-6    # 5× higher than v1
DECODER_LR = 2e-4    # 4× higher than v1

BATCH_SIZE  = 4
NUM_EPOCHS  = 20     # more epochs now that LR can actually make progress
IMAGE_SIZE  = 1024

MAX_NODATA_PCT = 0.20

# FIX: set to 0 — erosion was removing thin Indian road lines entirely.
# Re-chipped TIFF masks have roads ~3-6px wide; eroding by 3px wipes them
# out completely, leaving the model with ~0% road pixels to learn from,
# which causes it to collapse to "predict all background" (99.8% acc, dice=0).
# Only re-enable erosion if your masks are confirmed to be very thick (>15px).
MASK_EROSION_PX = 0

# Raised pos_weight since road pixels are genuinely rare in these chips
# (many chips from large TIFFs have sparse road coverage).
POS_WEIGHT = 13.0


# ============================================================
# 1. TILE QUALITY FILTER
# ============================================================
def compute_nodata_pct(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return 1.0
    black = ((img[:,:,0]==0) & (img[:,:,1]==0) & (img[:,:,2]==0))
    return float(black.mean())


def filter_tiles(image_dir, max_nodata_pct=0.20):
    all_files = sorted(f for f in os.listdir(image_dir)
                       if f.lower().endswith(('.jpg','.png','.tif')))
    kept, dropped = [], []
    for fname in all_files:
        pct = compute_nodata_pct(os.path.join(image_dir, fname))
        if pct <= max_nodata_pct:
            kept.append(fname)
        else:
            dropped.append((fname, pct))
    print(f"Tile filter: {len(kept)} kept / {len(dropped)} dropped "
          f"(threshold: {max_nodata_pct*100:.0f}% nodata)")
    return kept


# ============================================================
# 2. NODATA-AWARE DATASET  (with optional mask thinning)
# ============================================================
class IndianRoadsDataset(Dataset):
    def __init__(self, image_dir, mask_dir, valid_files=None,
                 transform=None, erosion_px=0):
        self.image_dir  = image_dir
        self.mask_dir   = mask_dir
        self.transform  = transform
        self.erosion_px = erosion_px
        self.images = valid_files if valid_files is not None else sorted(
            f for f in os.listdir(image_dir)
            if f.lower().endswith(('.jpg','.png','.tif')))

        if erosion_px > 0:
            self.erode_kernel = cv2.getStructuringElement(
                cv2.MORPH_ELLIPSE, (2*erosion_px+1, 2*erosion_px+1))
        else:
            self.erode_kernel = None

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        fname = self.images[index]

        # Handle _sat → _mask naming convention
        mask_fname = fname
        for src, dst in [('.jpg','.png'), ('.tif','.png'),
                         ('_sat','_mask'), ('sat','mask')]:
            mask_fname = mask_fname.replace(src, dst)

        img = np.array(Image.open(
            os.path.join(self.image_dir, fname)).convert("RGB"))
        mask = np.array(Image.open(
            os.path.join(self.mask_dir, mask_fname)).convert("L"),
            dtype=np.float32)

        # ========================================================
        # FIX: Ensure mask dimensions strictly match image dimensions
        # GIS exports can occasionally be off by 1 pixel (e.g. 1025x1025)
        # ========================================================
        if mask.shape != img.shape[:2]:
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)

        # nodata mask: pixels where all RGB channels are exactly 0
        nodata = ((img[:,:,0]==0) & (img[:,:,1]==0) &
                  (img[:,:,2]==0)).astype(np.float32)

        # normalise mask to [0,1]
        if mask.max() > 1.0:
            mask = mask / 255.0

        # FIX: thin thick QGIS polygon masks to match DeepGlobe style
        if self.erode_kernel is not None:
            mask_u8 = (mask * 255).astype(np.uint8)
            mask_u8 = cv2.erode(mask_u8, self.erode_kernel, iterations=1)
            mask = mask_u8.astype(np.float32) / 255.0

        if self.transform is not None:
            aug     = self.transform(image=img, masks=[mask, nodata])
            img     = aug["image"]
            mask    = aug["masks"][0]
            nodata  = aug["masks"][1]

        return img, mask, nodata


# ============================================================
# 3. NODATA-AWARE FOCAL TVERSKY LOSS
# ============================================================
class NodataAwareFocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=0.75,
                 smooth=1, pos_weight=13.0):
        super().__init__()
        self.alpha  = alpha
        self.beta   = beta
        self.gamma  = gamma
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([pos_weight]).to(DEVICE),
            reduction='none')

    def forward(self, inputs, targets, nodata_mask):
        valid = 1.0 - nodata_mask

        # BCE — zero out nodata pixels before averaging
        bce_loss = (self.bce(inputs, targets) * valid).sum() / (valid.sum() + 1e-8)

        # Focal Tversky — over valid pixels only
        probs = torch.sigmoid(inputs)
        p = probs[valid.bool()].view(-1)
        t = targets[valid.bool()].view(-1)

        TP = (p * t).sum()
        FP = ((1-t) * p).sum()
        FN = (t * (1-p)).sum()

        tversky      = (TP + self.smooth) / (TP + self.alpha*FP + self.beta*FN + self.smooth)
        focal_tversky = (1 - tversky) ** self.gamma

        return (0.5 * bce_loss) + focal_tversky


# ============================================================
# 4. UTILITIES
# ============================================================
def save_checkpoint(state, filename):
    torch.save(state, filename)
    print(f"  => Saved: {filename}")


def load_checkpoint(path, model):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    print(f"Loaded checkpoint: {path}")
    return model


def check_accuracy(loader, model, device):
    model.eval()
    num_correct = num_pixels = 0
    dice_score = 0.0

    with torch.no_grad():
        for x, y, nd in loader:
            x  = x.to(device)
            y  = y.to(device).unsqueeze(1)
            nd = nd.to(device).unsqueeze(1)
            valid = (1.0 - nd).bool()

            with torch.amp.autocast('cuda'):
                preds = (torch.sigmoid(model(x)) > 0.5).float()

            num_correct += (preds[valid] == y[valid]).sum().item()
            num_pixels  += valid.sum().item()
            dice_n = (2 * (preds * y * valid.float()).sum())
            dice_d = ((preds + y) * valid.float()).sum() + 1e-8
            dice_score += (dice_n / dice_d).item()

    acc  = num_correct / num_pixels * 100
    dice = dice_score / len(loader)
    print(f"  Acc (valid px): {acc:.2f}%")
    print(f"  Dice score:     {dice:.4f}")
    model.train()
    return dice


def train_fn(loader, model, optimizer, loss_fn, scaler):
    loop = tqdm(loader)
    for data, targets, nodata in loop:
        data    = data.to(DEVICE)
        targets = targets.float().unsqueeze(1).to(DEVICE)
        nodata  = nodata.float().unsqueeze(1).to(DEVICE)

        with torch.amp.autocast('cuda'):
            preds = model(data)
            loss  = loss_fn(preds, targets, nodata)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loop.set_postfix(loss=loss.item())


# ============================================================
# 5. MAIN
# ============================================================
def main():
    pp        = smp.encoders.get_preprocessing_params(ENCODER_NAME, pretrained=ENCODER_WEIGHTS)
    norm_mean = pp["mean"]
    norm_std  = pp["std"]

    train_transform = A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE,
                  interpolation=cv2.INTER_LINEAR,
                  mask_interpolation=cv2.INTER_NEAREST),
        A.Rotate(limit=35, p=1.0,
                  interpolation=cv2.INTER_LINEAR,
                  mask_interpolation=cv2.INTER_NEAREST),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.1),
        A.RandomBrightnessContrast(p=0.3),
        A.Normalize(mean=norm_mean, std=norm_std, max_pixel_value=255.0),
        ToTensorV2(),
    ])

    val_transform = A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE,
                  interpolation=cv2.INTER_LINEAR,
                  mask_interpolation=cv2.INTER_NEAREST),
        A.Normalize(mean=norm_mean, std=norm_std, max_pixel_value=255.0),
        ToTensorV2(),
    ])

    print("=== Filtering tiles by nodata % ===")
    train_valid = filter_tiles(TRAIN_IMG_DIR, MAX_NODATA_PCT)
    val_valid   = filter_tiles(VAL_IMG_DIR,   MAX_NODATA_PCT)

    train_ds = IndianRoadsDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR,
                                   valid_files=train_valid,
                                   transform=train_transform,
                                   erosion_px=MASK_EROSION_PX)
    val_ds   = IndianRoadsDataset(VAL_IMG_DIR, VAL_MASK_DIR,
                                   valid_files=val_valid,
                                   transform=val_transform,
                                   erosion_px=MASK_EROSION_PX)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                               shuffle=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE,
                               shuffle=False, num_workers=2, pin_memory=True)

    print(f"\nTraining on {len(train_ds)} tiles | Validating on {len(val_ds)} tiles")
    if MASK_EROSION_PX > 0:
        print(f"Mask erosion: {MASK_EROSION_PX}px (thinning thick polygon masks)")

    model = smp.Unet(
        encoder_name=ENCODER_NAME,
        encoder_weights=ENCODER_WEIGHTS,
        in_channels=3, classes=1,
    ).to(DEVICE)
    model = load_checkpoint(PRETRAINED_CKPT, model)

    loss_fn = NodataAwareFocalTverskyLoss(alpha=0.3, beta=0.7,
                                           gamma=0.75, pos_weight=POS_WEIGHT)

    optimizer = optim.Adam([
        {'params': model.encoder.parameters(),           'lr': ENCODER_LR},
        {'params': model.decoder.parameters(),           'lr': DECODER_LR},
        {'params': model.segmentation_head.parameters(), 'lr': DECODER_LR},
    ])

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3)

    scaler    = torch.amp.GradScaler('cuda')
    best_dice = 0.0

    print("\n=== Baseline: Indian val set BEFORE fine-tuning ===")
    check_accuracy(val_loader, model, DEVICE)

    for epoch in range(NUM_EPOCHS):
        print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
        train_fn(train_loader, model, optimizer, loss_fn, scaler)

        current_dice = check_accuracy(val_loader, model, DEVICE)
        
        # Track previous LR to mimic the removed 'verbose' functionality
        prev_lr = optimizer.param_groups[1]['lr']
        scheduler.step(current_dice)
        new_lr = optimizer.param_groups[1]['lr']
        
        if new_lr < prev_lr:
            print(f"  [Scheduler] Learning rate reduced to {new_lr:.2e}")

        save_checkpoint(
            {"state_dict": model.state_dict(), "optimizer": optimizer.state_dict()},
            filename="/kaggle/working/indian_latest.pth.tar")

        if current_dice > best_dice:
            best_dice = current_dice
            save_checkpoint(
                {"state_dict": model.state_dict(), "optimizer": optimizer.state_dict()},
                filename="/kaggle/working/indian_best.pth.tar")
            print(f"  ★ New best dice: {best_dice:.4f}")

    print(f"\nFine-tuning complete. Best Indian-dataset dice: {best_dice:.4f}")

if __name__ == "__main__":
    main()

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

=== Filtering tiles by nodata % ===
Tile filter: 229 kept / 0 dropped (threshold: 20% nodata)
Tile filter: 81 kept / 0 dropped (threshold: 20% nodata)

Training on 229 tiles | Validating on 81 tiles


model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Loaded checkpoint: /kaggle/input/notebooks/amrenderpalarwal/road-resilience-model-pretrained-encoder-unet-resn/my_checkpoint.pth.tar

=== Baseline: Indian val set BEFORE fine-tuning ===
  Acc (valid px): 91.81%
  Dice score:     0.2917

Epoch [1/20]


100%|██████████| 58/58 [00:26<00:00,  2.17it/s, loss=1.35]


...

  Acc (valid px): 90.65%
  Dice score:     0.2986
  => Saved: /kaggle/working/indian_latest.pth.tar

Epoch [19/20]


100%|██████████| 58/58 [00:29<00:00,  1.97it/s, loss=1.04]


  Acc (valid px): 90.47%
  Dice score:     0.3101
  [Scheduler] Learning rate reduced to 2.50e-05
  => Saved: /kaggle/working/indian_latest.pth.tar

Epoch [20/20]


100%|██████████| 58/58 [00:29<00:00,  1.96it/s, loss=1.31]


  Acc (valid px): 91.57%
  Dice score:     0.2985
  => Saved: /kaggle/working/indian_latest.pth.tar

Fine-tuning complete. Best Indian-dataset dice: 0.3143


In [6]:
# Quick diagnostic — run in a separate cell before restarting training
import cv2, os, numpy as np

mask_dir = "/kaggle/working/train/masks/"
files = os.listdir(mask_dir)[:10]
for f in files:
    m = cv2.imread(os.path.join(mask_dir, f), cv2.IMREAD_GRAYSCALE)
    road_pct = (m > 127).mean() * 100
    # Find typical road pixel width via distance transform
    dist = cv2.distanceTransform((m > 127).astype(np.uint8), cv2.DIST_L2, 5)
    avg_half_width = dist[dist > 0].mean() if (dist > 0).any() else 0
    print(f"{f}: road={road_pct:.1f}%  avg_half_width={avg_half_width:.1f}px  "
          f"(full width ~{avg_half_width*2:.0f}px)")

koramangala_256_256_mask.png: road=6.7%  avg_half_width=1.9px  (full width ~4px)
koramangala_1280_512_mask.png: road=4.8%  avg_half_width=1.9px  (full width ~4px)
Bellandur_1024_256_mask.png: road=10.6%  avg_half_width=1.9px  (full width ~4px)
koramangala_1280_0_mask.png: road=1.1%  avg_half_width=1.9px  (full width ~4px)
koramangala_768_1280_mask.png: road=9.6%  avg_half_width=1.9px  (full width ~4px)
koramangala_256_1536_mask.png: road=7.1%  avg_half_width=1.9px  (full width ~4px)
koramangala_512_2304_mask.png: road=9.9%  avg_half_width=1.9px  (full width ~4px)
Bellandur_0_1280_mask.png: road=3.2%  avg_half_width=1.9px  (full width ~4px)
Yelahanka_256_512_mask.png: road=8.1%  avg_half_width=2.0px  (full width ~4px)
indira_512_768_mask.png: road=6.3%  avg_half_width=2.0px  (full width ~4px)


In [7]:
import torch, cv2, numpy as np
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os

DEVICE = "cuda"
model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet",
                 in_channels=3, classes=1).to(DEVICE)
ckpt = torch.load("/kaggle/input/notebooks/amrenderpalarwal/topological-cons-finetuning-indian-data-banglore/indian_best.pth.tar", map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
model.eval()

pp = smp.encoders.get_preprocessing_params("resnet34", pretrained="imagenet")
transform = A.Compose([
    A.Resize(1024, 1024),
    A.Normalize(mean=pp["mean"], std=pp["std"], max_pixel_value=255.0),
    ToTensorV2(),
])

val_img_dir  = "/kaggle/working/val/images/"
val_mask_dir = "/kaggle/working/val/masks/"
files = sorted(os.listdir(val_img_dir))[:4]

for i, fname in enumerate(files):
    import numpy as np
    from PIL import Image

    img  = np.array(Image.open(os.path.join(val_img_dir, fname)).convert("RGB"))
    mname = fname.replace(".jpg",".png").replace("_sat","_mask")
    mask = np.array(Image.open(os.path.join(val_mask_dir, mname)).convert("L"))

    t = transform(image=img)["image"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = (torch.sigmoid(model(t)) > 0.5).float().squeeze().cpu().numpy()

    # Side by side: image | prediction | ground truth
    img_bgr  = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    pred_vis = (pred * 255).astype(np.uint8)
    mask_vis = mask

    panel = np.concatenate([
        cv2.resize(img_bgr,   (512, 512)),
        cv2.cvtColor(cv2.resize(pred_vis, (512, 512)), cv2.COLOR_GRAY2BGR),
        cv2.cvtColor(cv2.resize(mask_vis, (512, 512)), cv2.COLOR_GRAY2BGR),
    ], axis=1)
    cv2.imwrite(f"/kaggle/working/indian_check_{i}.png", panel)
    print(f"{fname}: road pixels in pred={pred.sum():.0f}  in mask={(mask>127).sum()}")

Hebbal_0_0_sat.jpg: road pixels in pred=130369  in mask=105930
Hebbal_0_256_sat.jpg: road pixels in pred=120020  in mask=127430
Hebbal_256_0_sat.jpg: road pixels in pred=154645  in mask=108852
Hebbal_256_256_sat.jpg: road pixels in pred=133940  in mask=118418
